### Schmucknachrichten – Lösung Schritt für Schritt

*43. Bundeswettbewerb Informatik (2024/2025), 2. Runde, Aufgabe 1*

Sybille und Pedram fädeln gerne Ketten aus bunten Glasperlen.

Dabei verstecken sie in den Ketten geheime Botschaften. Buchstaben werden durch Sequenzen aus farbigen Perlen dargestellt.
Um eine möglichst lange Botschaft in den Ketten unterzubringen, werden dafür häufige Buchstaben mit weniger Perlen codiert als
seltene Buchstaben.

Die beiden erstellen auf Wunsch auch für andere Personen Ketten. Die dabei verwendete Codetabelle wird in einem
„Beipackzettel“ mitgeschickt, damit man die Botschaften auch decodieren kann.

Mit dem folgenden Beipackzettel kommt man zum Beispiel für die Botschaft DIE SONNE SOLL DIR IMMER SCHEINEN auf genau
113 Perlen. Mit einem Durchmesser von 1mm pro Perle kann man die Botschaft also in 11,3 cm unterbringen.

(Bild auf dem Aufgabenblatt: Beipackzettel mit der Codetabelle)

Die beiden erhalten von den Interessenten mehrere zu codierende Sätze. Auf Grundlage dieser Sätze wird eine Tabelle mit einem
präfixfreien Code¹ erstellt, mit dem diese Sätze in einer möglichst kurzen Kette dargestellt werden können.

#### Aufgabe

a) Sybille und Pedram möchten die Botschaften mit Hilfe von mehr als zwei Farben noch kürzer auffädeln. Alle Perlen haben
denselben Durchmesser.

Schreibe ein Programm, das die Anzahl der Perlenfarben und einen Text mit einer Botschaft einliest, die aus beliebigen
Unicode-Symbolen bestehen kann. Anschließend gibt das Programm eine für diese Botschaft optimierte Codetabelle aus. Außerdem
soll die Gesamtlänge der Botschaft ausgegeben werden. Die Gesamtlänge soll möglichst klein sein!

b) Die Perlenfirma hat die Bestellung nicht ordnungsgemäß ausgeführt. Die Perlen verschiedener Farben haben auch
unterschiedliche Durchmesser. Ändere Dein Programm so, dass die Ketten aus diesen Perlen möglichst kurz sind.

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest, und dokumentiere die Ergebnisse.

¹ In einem präfixfreien Code gibt es kein Codewort, das Präfix eines anderen Codewortes ist. Dadurch ist garantiert, dass jede
Botschaft eindeutig dekodiert werden kann.

#### Beispieldaten

Im Ordner `beispieldaten` stehen `schmuck0.txt`, `schmuck00.txt` und `schmuck01.txt` (gleich große Perlen, Teil a) sowie
`schmuck1.txt` bis `schmuck9.txt` (verschieden große Perlen, Teil b). Jede Datei hat drei Zeilen:

1. die Zahl der Perlenarten,
2. die Durchmesser der Perlenarten in mm,
3. die Botschaft (Zeilenumbrüche sind durch Leerzeichen ersetzt; jedes Unicode-Zeichen, auch das Leerzeichen, wird codiert).

`schmuck0.txt` ist das Beispiel vom Aufgabenblatt:

```
2
1 1
DIE SONNE SOLL DIR IMMER SCHEINEN
```

Im README steht zur Kontrolle: Ein brauchbares Programm kommt bei `schmuck5.txt` auf 3162 mm und bei `schmuck9.txt` auf 36597 mm.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Präfixfreier Code als Baum:** Jedes Codewort ist ein Weg von der Wurzel zu einem **Blatt**. An jedem Knoten geht für jede
  Perlenart eine Kante ab. Weil die Zeichen nur an den Blättern sitzen, ist kein Codewort der Anfang eines anderen. Die Länge
  eines Codeworts in mm ist die Summe der Durchmesser auf dem Weg.
- **Gesamtlänge:** Summe über alle Zeichen von *Häufigkeit · Länge des Codeworts*. Daraus folgt sofort: Die **häufigsten Zeichen
  bekommen die kürzesten Codewörter**. Hat man die Längen der Codewörter, ist die Zuordnung also klar.
- **Huffman-Codierung:** Immer die seltensten Zeichen zu einem Knoten zusammenfassen. Bei zwei Perlen gleicher Größe ist das der
  bekannte Huffman-Code, bei `p` gleich großen Perlen fasst man jeweils `p` Knoten zusammen. Das Ergebnis ist **optimal**.
- **Kürzeste Wege (Dijkstra) und A\*:** Bei verschieden großen Perlen ist Huffman nicht mehr optimal. Wir bauen den Baum
  Millimeter für Millimeter auf und beschreiben jeden halbfertigen Baum durch wenige Zahlen (eine **Signatur**). Der beste Baum ist
  dann ein kürzester Weg in einem Graphen von Signaturen. A\* ist Dijkstra mit einer **Schätzung** der Restkosten.

**Nützliche Funktionen und Bibliotheken**
- `collections.Counter` zum Zählen, `most_common()` liefert die Zeichen nach Häufigkeit sortiert
- `heapq` (Prioritätswarteschlange: `heappush`, `heappop`, `heapify`)
- `dict` mit Tupeln als Schlüssel, `str.startswith`

## Die Idee

1. Die Zeichen der Botschaft **zählen**.
2. Eine **Codetabelle prüfen** können: präfixfrei? Wie lang wird die Kette? Kommt beim Decodieren die Botschaft heraus?
3. **Teil a)** mit der Huffman-Codierung für `p` gleich große Perlen lösen. Das ist optimal.
4. **Teil b), einfach:** Dieselbe Huffman-Codierung, aber bei jedem Knoten bekommt das häufigste Kind die kleinste Perle. Danach
   **nachbessern**: Die kürzesten Codewörter gehen an die häufigsten Zeichen. Das ist schnell, aber nicht immer optimal.
5. **Teil b), exakt:** Den Baum Millimeter für Millimeter aufbauen und mit Dijkstra und A\* den besten Baum suchen.
6. Alle Beispiele rechnen und mit den Lösungshinweisen vergleichen.

## Schritt 1: Einlesen und Zeichen zählen

`lies` gibt die Liste der Durchmesser und die Botschaft zurück. `Counter(text)` zählt, wie oft jedes Zeichen vorkommt, und
`most_common()` liefert die Paare `(Zeichen, Häufigkeit)`, das häufigste zuerst. Bei gleicher Häufigkeit bleibt die Reihenfolge des
ersten Auftretens. Damit man das Leerzeichen sieht, zeigt `zeige` es als `␣`.

In [ ]:
from collections import Counter


def lies(name):
    with open('beispieldaten/' + name + '.txt', encoding='utf-8') as f:
        zeilen = f.read().split('\n')
    durchmesser = [int(x) for x in zeilen[1].split()]
    text = zeilen[2]
    return durchmesser, text


def zeige(zeichen):
    if zeichen == ' ':
        return '␣'
    return zeichen


durchmesser, text = lies('schmuck0')
print('Durchmesser:', durchmesser)
print('Botschaft:', text, '(' + str(len(text)) + ' Zeichen)')
for zeichen, haeufigkeit in Counter(text).most_common():
    print(zeige(zeichen), haeufigkeit, end='   ')
print()

Durchmesser: [1, 1]
Botschaft: DIE SONNE SOLL DIR IMMER SCHEINEN (33 Zeichen)
E 5   ␣ 5   I 4   N 4   S 3   D 2   O 2   L 2   R 2   M 2   C 1   H 1   


Die Botschaft hat 33 Zeichen, davon 12 verschiedene. Am häufigsten sind `E` und das Leerzeichen (je 5-mal).

## Schritt 2: Eine Codetabelle prüfen

Eine **Codetabelle** ist ein `dict` von Zeichen auf Codewörter. Ein Codewort schreiben wir als Text aus den Namen der Perlenarten:
`'1'` ist die erste Perlenart der Datei, `'2'` die zweite und so weiter (die zehnte heißt `'A'`, so machen es auch die
Lösungshinweise).

- `wortlaenge` addiert die Durchmesser der Perlen eines Codeworts, `kettenlaenge` die Längen aller Codewörter der Botschaft.
- `ist_praefixfrei` sortiert die Codewörter. Ist ein Wort der Anfang eines anderen, steht in der sortierten Liste direkt dahinter
  ein Wort, das mit ihm anfängt (wie im Wörterbuch: nach „Ball“ kommt „Ballon“ vor „Bank“). Es genügt also, Nachbarn zu vergleichen.
- `decodiere` liest die Perlen einzeln und sammelt sie, bis die gesammelten Perlen ein Codewort sind. Das geht nur, weil der Code
  präfixfrei ist.

Zum Ausprobieren nehmen wir den Beipackzettel für `schmuck0.txt` aus den Lösungshinweisen (dort ist `1` eine rote und `2` eine blaue
Perle).

In [ ]:
PERLEN = '123456789A'                                      # Namen der Perlenarten: die erste heißt '1', die zehnte 'A'


def wortlaenge(wort, durchmesser):
    laenge = 0
    for perle in wort:
        laenge = laenge + durchmesser[PERLEN.index(perle)]
    return laenge


def kettenlaenge(text, tabelle, durchmesser):
    laenge = 0
    for zeichen in text:
        laenge = laenge + wortlaenge(tabelle[zeichen], durchmesser)
    return laenge


def ist_praefixfrei(tabelle):
    woerter = sorted(tabelle.values())
    for i in range(len(woerter) - 1):
        if woerter[i + 1].startswith(woerter[i]):          # gleich oder der Anfang des nächsten Worts
            return False
    return True


def codiere(text, tabelle):
    teile = []
    for zeichen in text:
        teile.append(tabelle[zeichen])
    return ''.join(teile)


def decodiere(kette, tabelle):
    zeichen_zu = {}                                        # umgekehrte Tabelle: Codewort -> Zeichen
    for zeichen in tabelle:
        zeichen_zu[tabelle[zeichen]] = zeichen
    ergebnis = []
    wort = ''
    for perle in kette:
        wort = wort + perle
        if wort in zeichen_zu:
            ergebnis.append(zeichen_zu[wort])
            wort = ''
    return ''.join(ergebnis)


beipackzettel = {'S': '211', 'R': '2122', 'H': '22222', 'M': '2211', 'E': '111', 'I': '122', 'D': '2121', ' ': '112',
                 'L': '2221', 'C': '22221', 'O': '2212', 'N': '121'}
durchmesser, text = lies('schmuck0')
kette = codiere(text, beipackzettel)
print('präfixfrei:', ist_praefixfrei(beipackzettel))
print('Kette:', kette)
print('Länge:', kettenlaenge(text, beipackzettel, durchmesser), 'mm')
print('decodiert:', decodiere(kette, beipackzettel))
print('Mit einem doppelten Wort präfixfrei?', ist_praefixfrei({'A': '11', 'B': '112', 'C': '2'}))

präfixfrei: True
Kette: 21211221111122112212121121111112211221222212221112212112221221121222211221111121221122112222122222111122121111121
Länge: 113 mm
decodiert: DIE SONNE SOLL DIR IMMER SCHEINEN
Mit einem doppelten Wort präfixfrei? False


Der Beipackzettel ist präfixfrei, die Kette ist 113 mm lang, und beim Decodieren kommt die Botschaft heraus. Bei der kleinen Tabelle
darunter ist `11` der Anfang von `112`, sie ist nicht präfixfrei.

## Schritt 3: Teil a) mit der Huffman-Codierung

Die Huffman-Codierung baut den Baum **von unten**: Jedes Zeichen ist zuerst ein eigener Baum mit seiner Häufigkeit als Gewicht.
Dann nimmt man immer die `p` leichtesten Bäume und hängt sie unter einen neuen Knoten. Sein Gewicht ist die Summe. Das wiederholt
man, bis nur noch ein Baum übrig ist. Seltene Zeichen werden früh zusammengefasst und landen deshalb tief im Baum, häufige oben.

**Füllzeichen:** Jeder Schritt macht aus `p` Bäumen einen, es werden also `p − 1` weniger. Damit am Ende genau ein Baum übrig
bleibt und der oberste Knoten alle `p` Perlen nutzt, muss `(Zahl der Bäume − 1)` durch `p − 1` teilbar sein. Wir fügen deshalb
Füllzeichen mit der Häufigkeit 0 hinzu. Sie werden als Erstes zusammengefasst und belegen die Plätze, die keiner braucht. Bei
zwei Perlen braucht man nie Füllzeichen.

**Umsetzung:** Die Bäume liegen in einem **Heap** (`heapq`) als Tupel `(gewicht, nummer, knoten)`. `heappop` holt immer das Tupel mit
dem kleinsten Gewicht. Die `nummer` ist nur dafür da, dass Python bei gleichem Gewicht nicht die Knoten vergleichen muss. Ein
`knoten` ist ein Zeichen (Blatt), `None` (Füllzeichen) oder eine Liste von Kindern.

Am Ende gehen wir den Baum von der Wurzel ab und hängen für das `i`-te Kind die `i`-te Perle an das Codewort. Die Kinder stehen nach
Gewicht absteigend in der Liste. Bei verschieden großen Perlen (Schritt 4) bekommt das schwerste Kind so die **kleinste** Perle, dazu
sortieren wir die Perlenarten nach ihrem Durchmesser. Paare `(durchmesser, nummer)` sortiert Python nach dem ersten Eintrag.

In [ ]:
import heapq


def huffman(durchmesser, text):
    p = len(durchmesser)
    paare = []
    for k in range(p):
        paare.append((durchmesser[k], k))
    paare.sort()                                           # Perlenarten nach Durchmesser sortiert
    perlen_sortiert = []
    for dm, k in paare:
        perlen_sortiert.append(PERLEN[k])

    heap = []
    for zeichen, haeufigkeit in Counter(text).most_common():
        heap.append((haeufigkeit, len(heap), zeichen))
    while (len(heap) - 1) % (p - 1) != 0:
        heap.append((0, len(heap), None))                  # Füllzeichen
    nummer = len(heap)
    heapq.heapify(heap)
    while len(heap) > 1:
        kinder = []
        gewicht = 0
        for i in range(p):
            g, nr, knoten = heapq.heappop(heap)            # die p leichtesten Bäume
            kinder.append(knoten)
            gewicht = gewicht + g
        kinder.reverse()                                   # das schwerste Kind zuerst, es bekommt die kleinste Perle
        heapq.heappush(heap, (gewicht, nummer, kinder))
        nummer = nummer + 1

    tabelle = {}
    offen = [(heap[0][2], '')]                             # (Knoten, Codewort bis dahin)
    while len(offen) > 0:
        knoten, wort = offen.pop()
        if type(knoten) == list:
            for i in range(len(knoten)):
                offen.append((knoten[i], wort + perlen_sortiert[i]))
        elif knoten is not None:
            tabelle[knoten] = wort
    return tabelle


def beipackzettel_zeigen(tabelle, text, hoechstens):
    """Die Codetabelle, das häufigste Zeichen zuerst, höchstens so viele Einträge."""
    eintraege = []
    for zeichen, haeufigkeit in Counter(text).most_common():
        eintraege.append(zeige(zeichen) + ' ' + tabelle[zeichen])
    for i in range(0, min(len(eintraege), hoechstens), 8):
        print('   '.join(eintraege[i:i + 8]))
    if len(eintraege) > hoechstens:
        print('... und', len(eintraege) - hoechstens, 'weitere Zeichen')


for name in ['schmuck0', 'schmuck00', 'schmuck01']:
    durchmesser, text = lies(name)
    tabelle = huffman(durchmesser, text)
    kette = codiere(text, tabelle)
    print(name + ':', len(durchmesser), 'Perlenarten,', len(tabelle), 'Zeichen, Länge', kettenlaenge(text, tabelle, durchmesser),
          'mm, präfixfrei:', ist_praefixfrei(tabelle), ', decodiert richtig:', decodiere(kette, tabelle) == text)
    beipackzettel_zeigen(tabelle, text, 24)
    print()

schmuck0: 2 Perlenarten, 12 Zeichen, Länge 113 mm, präfixfrei: True , decodiert richtig: True
E 112   ␣ 111   I 221   N 212   S 222   D 2112   O 2111   L 1222
R 1221   M 1212   C 12112   H 12111

schmuck00: 3 Perlenarten, 28 Zeichen, Länge 372 mm, präfixfrei: True , decodiert richtig: True
␣ 12   e 21   i 22   t 31   n 33   s 113   h 112   c 131
l 133   a 233   r 232   d 1112   D 1111   u 323   w 2311   m 1323
G 1322   g 1321   E 11132   k 11131   f 3223   W 3222   Z 3221   P 3213
... und 4 weitere Zeichen

schmuck01: 5 Perlenarten, 45 Zeichen, Länge 1150 mm, präfixfrei: True , decodiert richtig: True
␣ 3   e 5   n 12   r 14   i 15   s 22   l 25   a 24
t 23   h 42   c 43   o 111   g 45   m 112   u 113   . 114
d 131   k 115   E 133   , 132   D 211   b 135   w 212   ü 214
... und 21 weitere Zeichen



Die Huffman-Codierung kommt auf **113 mm**, **372 mm** und **1150 mm**. Das sind genau die optimalen Längen aus den
Lösungshinweisen. Für gleich große Perlen ist der Huffman-Code **beweisbar optimal**: Die seltensten Zeichen dürfen immer
Geschwister ganz unten im Baum sein, und fasst man sie zu einem Zeichen zusammen, bleibt eine kleinere Aufgabe derselben Art.
Teil a) ist damit gelöst. (Bei `schmuck0` kommt eine andere Tabelle heraus als in den Lösungshinweisen, aber dieselbe Länge. Es gibt
viele optimale Tabellen.)

## Schritt 4: Teil b), einfach – Huffman mit verschieden großen Perlen

`huffman` sortiert die Perlen schon nach Größe: Unter jedem Knoten bekommt das schwerste Kind die kleinste Perle. Das klingt
vernünftig, ist aber nicht mehr optimal. Huffman weiß nicht, dass die Perlen verschieden groß sind, die **Form** des Baums hängt nur
von den Häufigkeiten ab.

Ein zweites Problem: Ein seltenes Zeichen kann ein kürzeres Codewort bekommen als ein häufiges, weil es unter einer kleinen Perle
hängt. Darum bessern wir nach: `nachbessern` sortiert die Codewörter nach ihrer Länge in mm und gibt das kürzeste dem häufigsten
Zeichen, das zweitkürzeste dem zweithäufigsten und so weiter. Das ist nie schlechter (Austauschargument: Hat ein häufigeres Zeichen
ein längeres Wort, spart Tauschen Perlen) und der Code bleibt präfixfrei, denn die Codewörter sind dieselben.

In [ ]:
def nachbessern(tabelle, text, durchmesser):
    paare = []
    for wort in tabelle.values():
        paare.append((wortlaenge(wort, durchmesser), wort))
    paare.sort()                                           # kürzeste Codewörter zuerst
    neu = {}
    i = 0
    for zeichen, haeufigkeit in Counter(text).most_common():
        neu[zeichen] = paare[i][1]
        i = i + 1
    return neu


BEISPIELE = ['schmuck0', 'schmuck00', 'schmuck01', 'schmuck1', 'schmuck2', 'schmuck3', 'schmuck4', 'schmuck5', 'schmuck6',
             'schmuck7', 'schmuck8', 'schmuck9']
huffman_ergebnis = {}
for name in BEISPIELE:
    durchmesser, text = lies(name)
    tabelle = huffman(durchmesser, text)
    besser = nachbessern(tabelle, text, durchmesser)
    assert ist_praefixfrei(besser) and decodiere(codiere(text, besser), besser) == text
    huffman_ergebnis[name] = (kettenlaenge(text, tabelle, durchmesser), kettenlaenge(text, besser, durchmesser), besser)
    print(name.ljust(10), 'Durchmesser', str(durchmesser).ljust(32), 'Huffman', str(huffman_ergebnis[name][0]).rjust(6),
          'mm, nachgebessert', str(huffman_ergebnis[name][1]).rjust(6), 'mm')

schmuck0   Durchmesser [1, 1]                           Huffman    113 mm, nachgebessert    113 mm
schmuck00  Durchmesser [1, 1, 1]                        Huffman    372 mm, nachgebessert    372 mm
schmuck01  Durchmesser [1, 1, 1, 1, 1]                  Huffman   1150 mm, nachgebessert   1150 mm
schmuck1   Durchmesser [1, 1, 2]                        Huffman    197 mm, nachgebessert    195 mm
schmuck2   Durchmesser [1, 5]                           Huffman    145 mm, nachgebessert    145 mm
schmuck3   Durchmesser [1, 2, 3]                        Huffman    279 mm, nachgebessert    279 mm
schmuck4   Durchmesser [1, 5]                           Huffman    154 mm, nachgebessert    154 mm
schmuck5   Durchmesser [1, 1, 2, 3, 4, 5, 6]            Huffman   4010 mm, nachgebessert   3485 mm
schmuck6   Durchmesser [1, 2, 3]                        Huffman    244 mm, nachgebessert    238 mm
schmuck7   Durchmesser [1, 1, 1, 1, 1, 1, 1, 2, 3, 4]   Huffman 153144 mm, nachgebessert 137173 mm
schmuck8  

Bei gleich großen Perlen (die ersten drei) ändert das Nachbessern nichts. Bei verschieden großen Perlen hilft es oft sehr: bei
`schmuck5` von 4010 auf 3485 mm, bei `schmuck7` von 153144 auf 137173 mm, bei `schmuck9` von 41056 auf 37929 mm. Nur bei `schmuck2`, `3`
und `4` ändert sich nichts. Wie gut das ist, sehen wir erst, wenn wir das Optimum kennen. Die Lösungshinweise nennen für `schmuck5`
3162 mm, wir sind also gut 10 % schlechter. Bei `schmuck2` (Perlen mit 1 und 5 mm) sind es 145 statt 135 mm: Huffman baut einen
ausgewogenen Baum, besser wäre eine lange Kette aus kleinen Perlen.

## Schritt 5: Teil b), exakt – den Baum Millimeter für Millimeter aufbauen

Wir bauen den Baum jetzt **von oben**. Stell dir vor, du gehst mit einem Lineal Millimeter für Millimeter von der Wurzel nach unten.
Auf jeder Höhe (jeder **Tiefe** in mm) endet eine Reihe von Wegen. Jedes Wegende, das genau auf dieser Tiefe liegt, ist ein **offener
Knoten**, und wir müssen entscheiden:

- Der Knoten wird ein **Codewort** (ein Blatt), oder
- er wird **verlängert**: Für jede Perlenart hängt ein neuer offener Knoten darunter, um den Durchmesser der Perle tiefer.

Was braucht man sich dabei merken? Nicht den ganzen Baum! Es genügt die **Signatur** `(m, offen)`:

- `m` ist die Zahl der Codewörter, die bis zur aktuellen Tiefe fertig sind,
- `offen` ist ein Tupel `(l1, l2, …, lP)`: `l1` offene Knoten liegen 1 mm tiefer, `l2` liegen 2 mm tiefer und so weiter bis zum größten
  Durchmesser `P`.

**Kosten:** Wenn wir einen Millimeter tiefer gehen, wird jedes Zeichen, das noch **kein** Codewort hat, 1 mm länger. Weil die häufigsten
Zeichen die kürzesten Codewörter bekommen, sind die `m` fertigen Codewörter die der `m` häufigsten Zeichen. Ein Schritt kostet also die
Summe der Häufigkeiten aller Zeichen ab Nummer `m` (`rest[m]`). So zählt jedes Zeichen genau so viele Millimeter, wie sein Codewort lang
ist.

**Ein Schritt:** Die `l1` Knoten der nächsten Tiefe kommen an. Davon verlängern wir `q` Stück (`q` von 0 bis `l1`), die anderen
`l1 − q` werden Codewörter. Aus `(l1, …, lP)` wird `(l2, …, lP, 0)`, und für jeden verlängerten Knoten kommen `anzahl[i]` Knoten in der
Tiefe `+i` dazu (`anzahl[i]` ist die Zahl der Perlenarten mit Durchmesser `i`).

**Kürzen:** Mehr als `n − m` offene Knoten braucht man nie (`n` ist die Zahl der verschiedenen Zeichen). `kuerze` streicht die
überzähligen, und zwar die **tiefsten**. So bleiben die Signaturen klein, und gleichwertige Bäume haben dieselbe Signatur.

**Start und Ziel:** Am Anfang ist nur die Wurzel da, und sie wird verlängert: Start ist `(0, (anzahl[1], …, anzahl[P]))` (gekürzt). Fertig
sind wir bei `m = n`. Eine Signatur ohne offene Knoten und mit `m < n` ist eine **Sackgasse**.

In [ ]:
def vorbereiten(durchmesser, text):
    haeufigkeiten = []
    for zeichen, haeufigkeit in Counter(text).most_common():
        haeufigkeiten.append(haeufigkeit)
    n = len(haeufigkeiten)
    rest = [0] * (n + 1)                                   # rest[m] = Summe der Häufigkeiten der Zeichen m, m + 1, ..., n - 1
    for m in range(n - 1, -1, -1):
        rest[m] = rest[m + 1] + haeufigkeiten[m]
    anzahl = [0] * (max(durchmesser) + 1)                  # anzahl[i] = Zahl der Perlenarten mit Durchmesser i
    for dm in durchmesser:
        anzahl[dm] = anzahl[dm] + 1
    return n, rest, anzahl


def kuerze(m, offen, n):
    frei = n - m
    neu = []
    for l in offen:                                        # von oben nach unten: die tiefsten werden gestrichen
        l = min(l, frei)
        neu.append(l)
        frei = frei - l
    return tuple(neu)


def nachfolger(signatur, anzahl, n):
    """Alle Signaturen 1 mm tiefer als Paare (neue Signatur, q)."""
    m, offen = signatur
    ergebnis = []
    for q in range(offen[0] + 1):                          # q der ankommenden Knoten werden verlängert
        neu_m = m + offen[0] - q
        neu = list(offen[1:]) + [0]
        for i in range(1, len(offen) + 1):
            neu[i - 1] = neu[i - 1] + q * anzahl[i]
        ergebnis.append(((neu_m, kuerze(neu_m, neu, n)), q))
    return ergebnis


durchmesser, text = lies('schmuck2')
n, rest, anzahl = vorbereiten(durchmesser, text)
start = (0, kuerze(0, tuple(anzahl[1:]), n))
print('schmuck2: n =', n, ', anzahl =', anzahl, ', rest =', rest)
print('Start:', start)
for neu, q in nachfolger(start, anzahl, n):
    print('q =', q, '->', neu, ', Kosten des Schritts:', rest[start[0]])

schmuck2: n = 9 , anzahl = [0, 1, 0, 0, 0, 1] , rest = [41, 8, 7, 6, 5, 4, 3, 2, 1, 0]
Start: (0, (1, 0, 0, 0, 1))
q = 0 -> (1, (0, 0, 0, 1, 0)) , Kosten des Schritts: 41
q = 1 -> (0, (1, 0, 0, 1, 1)) , Kosten des Schritts: 41


Bei `schmuck2.txt` gibt es eine Perle mit 1 mm und eine mit 5 mm, also ist `P = 5` und `offen` hat fünf Einträge. Am Start liegt ein
offener Knoten 1 mm tief und einer 5 mm tief. Beim ersten Schritt kommt der 1-mm-Knoten an: Er wird Codewort (`q = 0`, dann ist
`m = 1`) oder verlängert (`q = 1`, dann liegen neue Knoten 1 mm und 5 mm tiefer). Der Schritt kostet in beiden Fällen 41 mm, denn
noch hat kein Zeichen ein Codewort, und die Botschaft hat 41 Zeichen.

## Schritt 6: Der kürzeste Weg mit Dijkstra

Jetzt ist es ein **Kürzeste-Wege-Problem**: Die Signaturen sind Knoten eines Graphen, jeder Schritt ist eine Kante mit den Kosten
`rest[m]`, und gesucht ist der billigste Weg vom Start zu einer Signatur mit `m = n`. Die Kosten sind nie negativ, also passt der
Algorithmus von **Dijkstra**: Ein Heap enthält Paare `(kosten, signatur)`. Wir holen immer die billigste Signatur heraus. Wenn sie
fertig ist (`m = n`), haben wir das Optimum, denn alle anderen Wege sind schon jetzt mindestens so teuer.

Dazu kommen drei Dinge:

- `kosten[signatur]` speichert die billigsten bisher bekannten Kosten. Liegt im Heap ein älterer, teurerer Eintrag, wird er übersprungen.
- `vorgaenger[signatur]` merkt sich, woher man kam (und mit welchem `q`), damit wir später den Baum zurückverfolgen können.
- `grenze` begrenzt die Zahl der untersuchten Signaturen, damit die Suche nicht zu lange läuft. Wird sie erreicht, gibt `suche`
  `None` zurück.

`suche` hat schon den Schalter `mit_schaetzung` für Schritt 7. Hier steht er auf `False`, die Schätzung ist dann 0.

In [ ]:
def suche(durchmesser, text, mit_schaetzung, grenze):
    """Ergebnis: (länge, untersucht, untere_schranke, weg). länge ist None, wenn die grenze erreicht wurde."""
    n, rest, anzahl = vorbereiten(durchmesser, text)
    start = (0, kuerze(0, tuple(anzahl[1:]), n))
    if mit_schaetzung:
        M = blaetter_bis(durchmesser, n)
    kosten = {start: 0}
    vorgaenger = {start: None}
    geschaetzt = {start: 0}                                # Schätzung der Restkosten (in Schritt 6 immer 0)
    heap = [(0, 0, start)]                                 # (kosten + schätzung, kosten, signatur)
    untersucht = 0
    while len(heap) > 0:
        f, k, signatur = heapq.heappop(heap)
        if k > kosten[signatur]:
            continue                                       # veralteter Eintrag
        m, offen = signatur
        if m == n:                                         # fertig: der kürzeste Weg
            weg = []
            while signatur is not None:
                if vorgaenger[signatur] is None:
                    weg.append((signatur, None))
                    signatur = None
                else:
                    weg.append((signatur, vorgaenger[signatur][1]))
                    signatur = vorgaenger[signatur][0]
            weg.reverse()
            return k, untersucht, k, weg
        if untersucht == grenze:
            return None, untersucht, f, None               # abgebrochen; f ist eine untere Schranke
        untersucht = untersucht + 1
        neu_k = k + rest[m]
        for neu, q in nachfolger(signatur, anzahl, n):
            if neu[0] < n and sum(neu[1]) == 0:
                continue                                   # Sackgasse
            if neu not in kosten or neu_k < kosten[neu]:
                kosten[neu] = neu_k
                vorgaenger[neu] = (signatur, q)
                if neu not in geschaetzt:
                    if mit_schaetzung:
                        geschaetzt[neu] = schaetzung(neu, rest, M, n)
                    else:
                        geschaetzt[neu] = 0
                heapq.heappush(heap, (neu_k + geschaetzt[neu], neu_k, neu))
    return None, untersucht, None, None


GRENZE = 20000
dijkstra_ergebnis = {}
for name in BEISPIELE:
    durchmesser, text = lies(name)
    laenge, untersucht, schranke, weg = suche(durchmesser, text, False, GRENZE)
    dijkstra_ergebnis[name] = (laenge, untersucht)
    if laenge is None:
        print(name.ljust(10), 'abgebrochen nach', untersucht, 'Signaturen')
    else:
        print(name.ljust(10), 'Länge', str(laenge).rjust(6), 'mm, untersuchte Signaturen:', untersucht)

schmuck0   Länge    113 mm, untersuchte Signaturen: 39
schmuck00  Länge    372 mm, untersuchte Signaturen: 84
schmuck01  Länge   1150 mm, untersuchte Signaturen: 84
schmuck1   Länge    191 mm, untersuchte Signaturen: 250
schmuck2   Länge    135 mm, untersuchte Signaturen: 232
schmuck3   Länge    279 mm, untersuchte Signaturen: 32
schmuck4   Länge    137 mm, untersuchte Signaturen: 1434
schmuck5   Länge   3162 mm, untersuchte Signaturen: 461
schmuck6   Länge    234 mm, untersuchte Signaturen: 1872
schmuck7   Länge 134559 mm, untersuchte Signaturen: 190
schmuck8   abgebrochen nach 20000 Signaturen
schmuck9   abgebrochen nach 20000 Signaturen


Bis auf zwei Beispiele findet Dijkstra das Optimum mit höchstens 1872 untersuchten Signaturen: **191, 135, 279, 137, 3162, 234 und
134559 mm**, genau die Werte der Lösungshinweise (und die 3162 mm aus dem README).

Bei `schmuck8` (321 verschiedene Zeichen) und `schmuck9` (674 verschiedene Zeichen) ist nach 20000 Signaturen Schluss. Ohne Grenze
schafft Dijkstra `schmuck8` auch, braucht aber viel länger (über 800000 Signaturen).

## Schritt 7: Schneller mit einer Schätzung (A\*)

Dijkstra untersucht auch viele Signaturen, die schon billig sind, aber nie zu einem guten Baum werden, zum Beispiel solche mit viel zu
wenigen offenen Knoten. **A\*** sortiert den Heap nicht nach den bisherigen Kosten, sondern nach *bisherige Kosten + Schätzung der
Restkosten*. Die Schätzung darf nie zu groß sein (sie muss eine **untere Schranke** sein), dann ist der erste fertige Baum, der aus
dem Heap kommt, immer noch optimal.

**Die Schätzung:** Wie viele Codewörter können aus einem offenen Knoten höchstens entstehen, die höchstens `s` mm tiefer enden?
Das ist `M[s]`: entweder der Knoten selbst (1), oder man verlängert ihn, dann zählt man für jede Perlenart mit Durchmesser `d ≤ s`
die Möglichkeiten `M[s − d]` zusammen. Daraus folgt für eine Signatur: Nach `t` weiteren Millimetern sind höchstens

`fertig = m + l1 · M[t − 1] + l2 · M[t − 2] + …` (nur die Summanden mit `i ≤ t`)

Zeichen fertig. Alle anderen kosten noch mindestens einen weiteren Schritt. `schaetzung` addiert diese Mindestkosten Schritt für
Schritt, bis alle `n` Zeichen fertig sein könnten.

`blaetter_bis` berechnet die Liste `M`, bis ein Eintrag mindestens `n` ist (größere Werte braucht man nicht).

In [ ]:
def blaetter_bis(durchmesser, n):
    M = []
    s = 0
    while len(M) == 0 or M[-1] < n:
        moeglich = 0
        for dm in durchmesser:
            if dm <= s:
                moeglich = moeglich + M[s - dm]
        M.append(max(1, moeglich))
        s = s + 1
    return M


def schaetzung(signatur, rest, M, n):
    m, offen = signatur
    summe = 0
    t = 0
    fertig = m
    while fertig < n:
        summe = summe + rest[fertig]                       # die noch nicht fertigen Zeichen werden 1 mm länger
        t = t + 1
        fertig = m
        for i in range(1, len(offen) + 1):
            if i <= t:
                fertig = fertig + offen[i - 1] * M[min(t - i, len(M) - 1)]
        fertig = min(fertig, n)
    return summe


durchmesser, text = lies('schmuck2')
n, rest, anzahl = vorbereiten(durchmesser, text)
print('M für schmuck2:', blaetter_bis(durchmesser, n))
start = (0, kuerze(0, tuple(anzahl[1:]), n))
print('Schätzung am Start:', schaetzung(start, rest, blaetter_bis(durchmesser, n), n))

astern_ergebnis = {}
for name in BEISPIELE:
    durchmesser, text = lies(name)
    astern_ergebnis[name] = suche(durchmesser, text, True, GRENZE)
    laenge, untersucht, schranke, weg = astern_ergebnis[name]
    if laenge is None:
        print(name.ljust(10), 'abgebrochen nach', untersucht, 'Signaturen, das Optimum ist mindestens', schranke, 'mm')
    else:
        print(name.ljust(10), 'Länge', str(laenge).rjust(6), 'mm, untersuchte Signaturen:', untersucht)

M für schmuck2: [1, 1, 1, 1, 1, 2, 3, 4, 5, 6, 8, 11]
Schätzung am Start: 99
schmuck0   Länge    113 mm, untersuchte Signaturen: 20
schmuck00  Länge    372 mm, untersuchte Signaturen: 37
schmuck01  Länge   1150 mm, untersuchte Signaturen: 11
schmuck1   Länge    191 mm, untersuchte Signaturen: 34
schmuck2   Länge    135 mm, untersuchte Signaturen: 18
schmuck3   Länge    279 mm, untersuchte Signaturen: 11
schmuck4   Länge    137 mm, untersuchte Signaturen: 19
schmuck5   Länge   3162 mm, untersuchte Signaturen: 35
schmuck6   Länge    234 mm, untersuchte Signaturen: 42
schmuck7   Länge 134559 mm, untersuchte Signaturen: 9
schmuck8   Länge   3287 mm, untersuchte Signaturen: 3499
schmuck9   abgebrochen nach 20000 Signaturen, das Optimum ist mindestens 35315 mm


Bei `schmuck2` ist `M = [1, 1, 1, 1, 1, 2, 3, 4, 5, 6, 8, 11]`: Mit Perlen von 1 und 5 mm kann ein Knoten erst ab 5 mm Tiefe mehr als ein
Codewort liefern. Die Schätzung am Start ist 99 mm, das Optimum ist 135 mm.

Mit der Schätzung braucht A\* viel weniger Signaturen als Dijkstra (bei `schmuck1` 34 statt 250, bei `schmuck6` 42 statt 1872), und jetzt ist auch **`schmuck8`** mit **3287 mm** nach 3499 Signaturen gelöst. Bei
**`schmuck9`** reicht es nicht: Nach 20000 Signaturen wissen wir nur, dass das Optimum **mindestens 35315 mm** ist. Mit der
Huffman-Codierung haben wir eine Kette mit 37929 mm, das Optimum liegt also dazwischen. Auch eine viel größere Grenze hilft hier
nicht, der Heap wird dann riesig. Die Lösungshinweise brauchen für `schmuck9` mit diesem Verfahren (und mehr Tricks) einige
Stunden. Das Optimum von 36597 mm bekommen wir mit MiniZinc (siehe unten).

## Schritt 8: Die Codetabelle aus dem Weg bauen

Der Weg sagt uns für jeden Millimeter, wie viele der ankommenden Knoten verlängert werden (`q`). Daraus bauen wir die Codewörter
nach: Wir spielen den Weg noch einmal ab und merken uns diesmal die **Codewörter** der offenen Knoten. `offen[i]` ist die Liste der
Präfixe, die `i` mm tiefer als die aktuelle Tiefe enden.

- Am Start hängt unter der Wurzel jede Perlenart: Die Perle `k` mit Durchmesser `d` kommt nach `offen[d]`.
- In jedem Schritt kommen die Präfixe aus `offen[1]` an. Die ersten `q` werden verlängert, die anderen werden Codewörter.
- Danach kürzen wir die Listen auf die Längen in der Signatur, genau wie `kuerze` (welche gleich tiefen Knoten gestrichen werden, ist
  egal).

Die Codewörter entstehen von oben nach unten, also kürzeste zuerst. Das erste bekommt das häufigste Zeichen und so weiter.

In [ ]:
def baue_tabelle(weg, durchmesser, text):
    P = max(durchmesser)
    offen = []
    for i in range(P + 1):
        offen.append([])
    for k in range(len(durchmesser)):
        offen[durchmesser[k]].append(PERLEN[k])            # unter der Wurzel hängt jede Perlenart
    woerter = []
    for schritt in range(len(weg)):
        signatur, q = weg[schritt]
        if schritt > 0:
            ankommend = offen[1]
            offen = [[]] + offen[2:] + [[]]                # alle offenen Knoten sind jetzt 1 mm näher
            for wort in ankommend[:q]:                     # verlängern
                for k in range(len(durchmesser)):
                    offen[durchmesser[k]].append(wort + PERLEN[k])
            for wort in ankommend[q:]:                     # werden Codewörter
                woerter.append(wort)
        m, anzahl_offen = signatur
        for i in range(1, P + 1):
            offen[i] = offen[i][:anzahl_offen[i - 1]]      # kürzen wie in der Signatur
    tabelle = {}
    i = 0
    for zeichen, haeufigkeit in Counter(text).most_common():
        tabelle[zeichen] = woerter[i]
        i = i + 1
    return tabelle


durchmesser, text = lies('schmuck2')
laenge, untersucht, schranke, weg = astern_ergebnis['schmuck2']
print('Weg (Signatur, q):')
for signatur, q in weg:
    print('  ', signatur, q)
tabelle = baue_tabelle(weg, durchmesser, text)
print('Länge:', kettenlaenge(text, tabelle, durchmesser), 'mm, präfixfrei:', ist_praefixfrei(tabelle),
      ', decodiert richtig:', decodiere(codiere(text, tabelle), tabelle) == text)
beipackzettel_zeigen(tabelle, text, 24)

Weg (Signatur, q):
   (0, (1, 0, 0, 0, 1)) None
   (1, (0, 0, 0, 1, 0)) 0
   (1, (0, 0, 1, 0, 0)) 0
   (1, (0, 1, 0, 0, 0)) 0
   (1, (1, 0, 0, 0, 0)) 0
   (1, (1, 0, 0, 0, 1)) 1
   (1, (1, 0, 0, 1, 1)) 1
   (1, (1, 0, 1, 1, 1)) 1
   (1, (1, 1, 1, 1, 1)) 1
   (1, (2, 1, 1, 1, 1)) 1
   (1, (3, 1, 1, 1, 2)) 2
   (4, (1, 1, 1, 2, 0)) 0
   (5, (1, 1, 2, 0, 0)) 0
   (6, (1, 2, 0, 0, 0)) 0
   (7, (2, 0, 0, 0, 0)) 0
   (9, (0, 0, 0, 0, 0)) 0
Länge: 135 mm, präfixfrei: True , decodiert richtig: True
a 1   ␣ 212   b 221   c 2111111   d 2112   e 21112   f 211112   g 222
h 2111112


Das ist der optimale Baum für `schmuck2`: Das `a` (33-mal) bekommt die kleine Perle allein, alle anderen Zeichen hängen unter der
großen Perle. Darunter geht es mit kleinen Perlen weiter. Die Kette ist 135 mm lang, so wie in den Lösungshinweisen.

## Schritt 9: Alle Beispiele

Für jedes Beispiel nehmen wir den Baum von A\*, wenn die Suche fertig wurde, sonst die nachgebesserte Huffman-Codierung. Jede Tabelle
wird geprüft (präfixfrei, Länge, Decodieren). `?` heißt: Wir kennen das Optimum mit Python nicht.

In [ ]:
beste = {}
for name in BEISPIELE:
    durchmesser, text = lies(name)
    laenge, untersucht, schranke, weg = astern_ergebnis[name]
    if laenge is None:
        tabelle = huffman_ergebnis[name][2]
        optimal = '?'
    else:
        tabelle = baue_tabelle(weg, durchmesser, text)
        assert kettenlaenge(text, tabelle, durchmesser) == laenge
        optimal = 'ja'
    assert ist_praefixfrei(tabelle) and decodiere(codiere(text, tabelle), tabelle) == text
    beste[name] = tabelle
    print(name.ljust(10), 'Huffman', str(huffman_ergebnis[name][1]).rjust(6), 'mm, beste Tabelle',
          str(kettenlaenge(text, tabelle, durchmesser)).rjust(6), 'mm, optimal:', optimal)

schmuck0   Huffman    113 mm, beste Tabelle    113 mm, optimal: ja
schmuck00  Huffman    372 mm, beste Tabelle    372 mm, optimal: ja
schmuck01  Huffman   1150 mm, beste Tabelle   1150 mm, optimal: ja
schmuck1   Huffman    195 mm, beste Tabelle    191 mm, optimal: ja
schmuck2   Huffman    145 mm, beste Tabelle    135 mm, optimal: ja
schmuck3   Huffman    279 mm, beste Tabelle    279 mm, optimal: ja
schmuck4   Huffman    154 mm, beste Tabelle    137 mm, optimal: ja
schmuck5   Huffman   3485 mm, beste Tabelle   3162 mm, optimal: ja
schmuck6   Huffman    238 mm, beste Tabelle    234 mm, optimal: ja
schmuck7   Huffman 137173 mm, beste Tabelle 134559 mm, optimal: ja
schmuck8   Huffman   3430 mm, beste Tabelle   3287 mm, optimal: ja
schmuck9   Huffman  37929 mm, beste Tabelle  37929 mm, optimal: ?


**Vergleich mit den Lösungshinweisen** (Tabelle 2, Kettenlängen in mm). „Greedy“ ist das Verfahren der Lösungshinweise (eine dynamische
Programmierung, die die sortierten Zeichen in Gruppen aufteilt) mit Nachbessern, „optimal“ die Spalte ILP/Polynomiell.

| Beispiel | Perlen (mm) | Zeichen | Huffman nachgebessert | A\* (Python) | Hinweise: Greedy | Hinweise: optimal |
|---|---|---|---|---|---|---|
| schmuck0 | 1 1 | 12 | 113 | **113** | 113 | 113 |
| schmuck00 | 1 1 1 | 28 | 372 | **372** | 372 | 372 |
| schmuck01 | 1 1 1 1 1 | 45 | 1150 | **1150** | 1150 | 1150 |
| schmuck1 | 1 1 2 | 25 | 195 | **191** | 191 | 191 |
| schmuck2 | 1 5 | 9 | 145 | **135** | 135 | 135 |
| schmuck3 | 1 2 3 | 9 | 279 | **279** | 285 | 279 |
| schmuck4 | 1 5 | 14 | 154 | **137** | 137 | 137 |
| schmuck5 | 1 1 2 3 4 5 6 | 41 | 3485 | **3162** | 3240 | 3162 |
| schmuck6 | 1 2 3 | 34 | 238 | **234** | 234 | 234 |
| schmuck7 | 1 1 1 1 1 1 1 2 3 4 | 82 | 137173 | **134559** | 134902 | 134559 |
| schmuck8 | 1 1 2 2 3 | 321 | 3430 | **3287** | 3300 | 3287 |
| schmuck9 | 1 2 3 4 | 674 | 37929 | ? (≥ 35315) | 36936 | 36597 |

- Bei **gleich großen Perlen** (Teil a) ist Huffman optimal.
- Bei **verschieden großen Perlen** findet A\* in elf von zwölf Beispielen das Optimum, alle Werte stimmen mit den Lösungshinweisen
  überein. Die einfache Huffman-Codierung ist bis zu gut 12 % schlechter (`schmuck4`: 154 statt 137 mm; `schmuck5`: gut 10 %), bei `schmuck3` ist sie zufällig
  optimal und sogar besser als das Greedy-Verfahren der Hinweise.
- Bei **`schmuck9`** bleibt das Python-Ergebnis offen. Unsere beste Kette (Huffman, 37929 mm) ist gut 3,6 % länger als das Optimum
  36597 mm, und auch länger als das Greedy der Hinweise (36936 mm).

**Die Beipackzettel.** Die ersten Einträge jeder Tabelle, das häufigste Zeichen zuerst (das Leerzeichen als `␣`):

In [ ]:
for name in BEISPIELE:
    durchmesser, text = lies(name)
    print(name + ':', kettenlaenge(text, beste[name], durchmesser), 'mm')
    beipackzettel_zeigen(beste[name], text, 16)
    print()

schmuck0: 113 mm
E 122   ␣ 211   I 212   N 221   S 222   D 1112   O 1121   L 1122
R 1211   M 1212   C 11111   H 11112

schmuck00: 372 mm
␣ 22   e 23   i 31   t 32   n 33   s 121   h 122   c 123
l 131   a 132   r 133   d 211   D 212   u 213   w 1121   m 1122
... und 12 weitere Zeichen

schmuck01: 1150 mm
␣ 4   e 5   n 15   r 21   i 22   s 23   l 24   a 25
t 31   h 32   c 33   o 34   g 35   m 114   u 115   . 121
... und 29 weitere Zeichen

schmuck1: 191 mm
e 22   t 111   ␣ 112   r 121   i 122   n 211   w 212   B 123
I 213   s 131   f 132   " 231   b 232   W 311   N 312   F 321
... und 9 weitere Zeichen

schmuck2: 135 mm
a 1   ␣ 212   b 221   c 2111111   d 2112   e 21112   f 211112   g 222
h 2111112

schmuck3: 279 mm
a 2   b 11   c 12   ␣ 31   d 32   e 131   f 33   g 132
h 133

schmuck4: 137 mm
a 1111111   b 1112   c 1121   d 1211   e 2111   f 11112   g 22   h 111112
i 122   j 212   k 1111112   l 1122   m 1212   n 2112

schmuck5: 3162 mm
␣ 21   e 22   t 4   i 13   o 23   s 31   a 32   n 1

Die Ausgabe in eine Datei schreibt man zum Beispiel so:

```python
with open('schmuck1_beipackzettel.txt', 'w', encoding='utf-8') as f:
    for zeichen, haeufigkeit in Counter(text).most_common():
        f.write(zeichen + ' ' + beste['schmuck1'][zeichen] + '\n')
```

## Für Neugierige

**Wie kurz kann es höchstens werden? Die Shannon-Schranke.** Aus der Informationstheorie kommt eine untere Schranke für jede eindeutig
decodierbare Codierung. Zuerst die **Kapazität** der Perlen: die Zahl `t` zwischen 0 und 1 mit `t^d1 + t^d2 + … + t^dp = 1`. Bei zwei
Perlen mit 1 mm ist `t = 0,5`, jede Perle trägt dann genau ein Bit. Eine Perle mit dem Durchmesser `d` trägt `−d · log2(t)` Bit.
Die **Entropie** `H = −Σ p_i · log2(p_i)` (mit den relativen Häufigkeiten `p_i`) ist die Zahl der Bits, die ein Zeichen im Mittel
mindestens braucht. Die Kette ist also mindestens `Zahl der Zeichen · H / (−log2 t)` mm lang. `t` finden wir mit **Bisektion**:
Die Summe der Potenzen wird kleiner, wenn `t` kleiner wird.

In [ ]:
import math


def kapazitaet(durchmesser):
    unten = 0.0
    oben = 1.0
    for schritt in range(60):
        t = (unten + oben) / 2
        summe = 0
        for dm in durchmesser:
            summe = summe + t ** dm
        if summe > 1:
            oben = t
        else:
            unten = t
    return unten


def shannon_schranke(durchmesser, text):
    entropie = 0
    for zeichen, haeufigkeit in Counter(text).most_common():
        p = haeufigkeit / len(text)
        entropie = entropie - p * math.log2(p)
    return len(text) * entropie / -math.log2(kapazitaet(durchmesser))


for name in BEISPIELE:
    durchmesser, text = lies(name)
    schranke = shannon_schranke(durchmesser, text)
    print(name.ljust(10), 'Shannon-Schranke', ('%.2f' % schranke).rjust(10), 'mm, aufgerundet', math.ceil(schranke), 'mm')

schmuck0   Shannon-Schranke     112.49 mm, aufgerundet 113 mm
schmuck00  Shannon-Schranke     362.41 mm, aufgerundet 363 mm
schmuck01  Shannon-Schranke    1107.83 mm, aufgerundet 1108 mm
schmuck1   Shannon-Schranke     187.60 mm, aufgerundet 188 mm
schmuck2   Shannon-Schranke     131.12 mm, aufgerundet 132 mm
schmuck3   Shannon-Schranke     252.83 mm, aufgerundet 253 mm
schmuck4   Shannon-Schranke     131.39 mm, aufgerundet 132 mm
schmuck5   Shannon-Schranke    3132.89 mm, aufgerundet 3133 mm
schmuck6   Shannon-Schranke     227.63 mm, aufgerundet 228 mm
schmuck7   Shannon-Schranke  129249.79 mm, aufgerundet 129250 mm
schmuck8   Shannon-Schranke    3231.28 mm, aufgerundet 3232 mm
schmuck9   Shannon-Schranke   36387.80 mm, aufgerundet 36388 mm


Die Lösungshinweise geben diese Schranken abgerundet an (112, 362, 1107, …, 36387 mm), und so kommt es auch hier heraus. Da Kettenlängen
ganze Zahlen sind, darf man **aufrunden**. Bei `schmuck0` ist die aufgerundete Schranke 113 mm, das beweist schon ohne Suche, dass
113 mm optimal sind. Bei `schmuck9` ist die Schranke mit 36388 mm viel besser als die 35315 mm von A\*, aber immer noch 209 mm vom
Optimum entfernt.

Man kann die Idee der Schranke auch in die **Schätzung von A\*** einbauen (für die noch offenen Zeichen und die offenen Knoten einer
Signatur). Dann braucht A\* noch weniger Signaturen, `schmuck9` bleibt aber auch damit zu groß für Python.

**Was die Lösungshinweise noch zeigen:**
- Ein **Greedy-Verfahren mit dynamischer Programmierung**: Die nach Häufigkeit sortierten Zeichen werden in Gruppen geteilt, jede Gruppe
  bekommt eine Perle als erste Perle, und so weiter in den Gruppen. Die beste Aufteilung findet man mit einer Tabelle über alle Intervalle
  (Laufzeit `O(p · n³)`).
- Das **ILP nach Karp** (1961): ein ganzzahliges lineares Programm über die Zahl der Codewörter und Präfixe je Länge. Das ist das Modell im
  MiniZinc-Abschnitt.
- Das Verfahren von **Golin und Rote** (1998), das wir in Schritt 5 bis 7 benutzt haben: Es ist polynomiell in `n`, aber exponentiell im
  größten Durchmesser (die Signatur hat `P` Einträge).
- **Arithmetische Codierung:** Gibt man die Präfixfreiheit auf, kann man unter die optimalen präfixfreien Längen kommen (bei `schmuck0`
  112 mm statt 113 mm). Dann codiert man die ganze Botschaft als eine Zahl, und von Hand decodieren kann man sie nicht mehr.

## Zum Schluss: dasselbe mit MiniZinc

Statt den Baum zu suchen, kann man auch beschreiben, **wie viele Codewörter jeder Länge** es gibt. Das ist das ganzzahlige lineare
Programm von Karp aus den Lösungshinweisen. Für jede Länge `j` (in mm, bis zu einer Obergrenze `D`) gibt es zwei Variablen:

- `wort[j]`: die Zahl der Codewörter der Länge `j`,
- `praefix[j]`: die Zahl der Präfixe (innerer Knoten) der Länge `j`. `praefix[0] = 1` ist die Wurzel.

Jeder Knoten der Länge `j` entsteht, wenn man an einen Präfix der Länge `j − k` eine Perle mit Durchmesser `k` hängt. Darum gilt
`wort[j] + praefix[j] <= Σ anzahl[k] · praefix[j − k]`. Das `<=` erlaubt Plätze, die keiner braucht. Mehr muss man über den Baum
nicht sagen: Zu jeder Folge von Zahlen, die diese Bedingungen erfüllt, gibt es einen passenden Baum.

Welches Zeichen welche Länge bekommt, sagt `x`. Zeichen mit **gleicher Häufigkeit** sind austauschbar. Darum fassen wir sie zusammen:
`x[i, j]` ist die Zahl der Zeichen mit der Häufigkeit `haeufigkeit[i]`, die ein Codewort der Länge `j` bekommen. Das spart sehr viele
Variablen, bei `schmuck9` gibt es 674 Zeichen, aber nur wenige verschiedene Häufigkeiten. Die Kettenlänge ist die Summe von
`Häufigkeit · Länge` über alle Zeichen.

```minizinc
int: P;                                   % größter Durchmesser
array[1..P] of int: anzahl;               % anzahl[k] = Zahl der Perlenarten mit Durchmesser k
int: g;                                   % Zahl der verschiedenen Häufigkeiten
array[1..g] of int: haeufigkeit;          % die Häufigkeiten
array[1..g] of int: zeichen;              % zeichen[i] = Zahl der Zeichen mit der Häufigkeit haeufigkeit[i]
int: D;                                   % das längste erlaubte Codewort (in mm)
int: n = sum(zeichen);                    % Zahl der verschiedenen Zeichen

% x[i, j]: so viele Zeichen mit der Häufigkeit haeufigkeit[i] bekommen ein Codewort der Länge j mm
array[1..g, 1..D] of var 0..n: x;
array[0..D] of var 0..n: praefix;         % Zahl der Präfixe (innere Knoten) der Länge j
array[1..D] of var 0..n: wort;            % Zahl der Codewörter der Länge j

constraint praefix[0] = 1;                % die Wurzel
constraint forall(i in 1..g)(sum(j in 1..D)(x[i, j]) = zeichen[i]);
constraint forall(j in 1..D)(wort[j] = sum(i in 1..g)(x[i, j]));
% Jeder Knoten der Länge j hängt an einem Präfix der Länge j - k unter einer Perle mit Durchmesser k
constraint forall(j in 1..D)(
  wort[j] + praefix[j] <= sum(k in 1..P where k <= j)(anzahl[k] * praefix[j - k]));

var int: laenge = sum(i in 1..g, j in 1..D)(haeufigkeit[i] * j * x[i, j]);
solve minimize laenge;

output ["laenge = \(laenge) mm\n", "codewoerter je Länge = \(wort)\n"];
```

Die Daten baut eine Python-Zelle aus der Beispieldatei. `D = 25` ist großzügig, kein optimales Codewort in den Beispielen ist länger als
15 mm (und mit `D = 40` kommt überall dasselbe heraus).

In [ ]:
def als_dzn(durchmesser, text, D):
    haeufigkeiten = list(Counter(text).values())
    werte = sorted(set(haeufigkeiten), reverse=True)
    P = max(durchmesser)
    anzahl = []
    for k in range(1, P + 1):
        anzahl.append(durchmesser.count(k))
    zeichen = []
    for w in werte:
        zeichen.append(haeufigkeiten.count(w))
    return 'P = %d;\nanzahl = %s;\ng = %d;\nhaeufigkeit = %s;\nzeichen = %s;\nD = %d;' % (
        P, anzahl, len(werte), werte, zeichen, D)


durchmesser, text = lies('schmuck0')
print(als_dzn(durchmesser, text, 25))

P = 1;
anzahl = [2];
g = 5;
haeufigkeit = [5, 4, 3, 2, 1];
zeichen = [2, 2, 1, 5, 2];
D = 25;


Die Codewörter sind bei `schmuck0` 3, 4 und 5 mm lang (fünf, fünf und zwei Stück):

**Ausgabe:**

```
laenge = 113 mm
codewoerter je Länge = [0, 0, 5, 5, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
----------
==========
```

Und das Beispiel, an dem unsere Python-Suche gescheitert ist, `schmuck9.txt`:

**Ausgabe:**

```
laenge = 36597 mm
codewoerter je Länge = [0, 0, 0, 0, 4, 14, 11, 20, 27, 74, 98, 181, 148, 97, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
----------
==========
```

Mit dem Solver **HiGHS** (einem Solver für lineare Programme, in MiniZinc unter „HiGHS“ auswählen) sind alle zwölf Beispiele schnell
gelöst, auch `schmuck9` mit den **36597 mm** aus dem README und den Lösungshinweisen. Alle Längen stimmen mit unseren Python-Ergebnissen
überein. Die Constraint-Solver kommen mit diesem Modell bei den größeren Beispielen nicht zurecht. Gecode und Chuffed finden bei `schmuck5`
in einer Minute nur Ketten mit über 14000 mm, CP-SAT lieferte nach mehreren Minuten gar kein Ergebnis. Lineare Programme mit großen Summen sind die Stärke
von HiGHS.

MiniZinc gibt nur die **Zahl** der Codewörter je Länge aus, nicht die Codewörter selbst. Die Tabelle baut man daraus wie in Schritt 8
nach: Tiefe für Tiefe so viele Präfixe verlängern, wie `praefix` sagt, und die anderen zu Codewörtern machen.